## imports

In [17]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

## import Data

In [18]:
district = pd.read_csv("../BankData/district.csv",delimiter=";")

district

,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,A11,A12,A13,A14,A15,A16
0,1,Hl.m. Praha,Prague,1204953,0,0,0,1,1,100.0,12541,0.29,0.43,167,85677,99107
1,2,Benesov,central Bohemia,88884,80,26,6,2,5,46.7,8507,1.67,1.85,132,2159,2674
2,3,Beroun,central Bohemia,75232,55,26,4,1,5,41.7,8980,1.95,2.21,111,2824,2813
3,4,Kladno,central Bohemia,149893,63,29,6,2,6,67.4,9753,4.64,5.05,109,5244,5892
4,5,Kolin,central Bohemia,95616,65,30,4,1,6,51.4,9307,3.85,4.43,118,2616,3040
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72,73,Opava,north Moravia,182027,17,49,12,2,7,56.4,8746,3.33,3.74,90,4355,4433
73,74,Ostrava - mesto,north Moravia,323870,0,0,0,1,1,100.0,10673,4.75,5.44,100,18782,18347
74,75,Prerov,north Moravia,138032,67,30,4,2,5,64.6,8819,5.38,5.66,99,4063,4505
75,76,Sumperk,north Moravia,127369,31,32,13,2,7,51.2,8369,4.73,5.88,107,3736,2807


## Check for nulls

In [19]:
district.isna().sum()

A1     0
A2     0
A3     0
A4     0
A5     0
A6     0
A7     0
A8     0
A9     0
A10    0
A11    0
A12    0
A13    0
A14    0
A15    0
A16    0
dtype: int64

In [20]:
district.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77 entries, 0 to 76
Data columns (total 16 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   A1      77 non-null     int64  
 1   A2      77 non-null     object 
 2   A3      77 non-null     object 
 3   A4      77 non-null     int64  
 4   A5      77 non-null     int64  
 5   A6      77 non-null     int64  
 6   A7      77 non-null     int64  
 7   A8      77 non-null     int64  
 8   A9      77 non-null     int64  
 9   A10     77 non-null     float64
 10  A11     77 non-null     int64  
 11  A12     77 non-null     object 
 12  A13     77 non-null     float64
 13  A14     77 non-null     int64  
 14  A15     77 non-null     object 
 15  A16     77 non-null     int64  
dtypes: float64(2), int64(10), object(4)
memory usage: 9.8+ KB


## Change column name as is in documnet

In [21]:
district_rename = {
    'A1': 'district_id',
    'A2': 'district_name',
    'A3': 'region',
    'A4': 'population',
    'A5': 'municipalities_lt_500',
    'A6': 'municipalities_500_1999',
    'A7': 'municipalities_2000_9999',
    'A8': 'municipalities_gt_10000',
    'A9': 'num_cities',
    'A10': 'urban_ratio',
    'A11': 'avg_salary',
    'A12': 'unemployment_rate_95',
    'A13': 'unemployment_rate_96',
    'A14': 'entrepreneurs_per_1000',
    'A15': 'crimes_95',
    'A16': 'crimes_96'
}

district = district.rename(columns=district_rename)
district.head()

,district_id,district_name,region,population,municipalities_lt_500,municipalities_500_1999,municipalities_2000_9999,municipalities_gt_10000,num_cities,urban_ratio,avg_salary,unemployment_rate_95,unemployment_rate_96,entrepreneurs_per_1000,crimes_95,crimes_96
0,1,Hl.m. Praha,Prague,1204953,0,0,0,1,1,100.0,12541,0.29,0.43,167,85677,99107
1,2,Benesov,central Bohemia,88884,80,26,6,2,5,46.7,8507,1.67,1.85,132,2159,2674
2,3,Beroun,central Bohemia,75232,55,26,4,1,5,41.7,8980,1.95,2.21,111,2824,2813
3,4,Kladno,central Bohemia,149893,63,29,6,2,6,67.4,9753,4.64,5.05,109,5244,5892
4,5,Kolin,central Bohemia,95616,65,30,4,1,6,51.4,9307,3.85,4.43,118,2616,3040


In [22]:
district['unemployment_rate_95'] = pd.to_numeric(district['unemployment_rate_95'].replace('?', np.nan), errors='coerce')
district['crimes_95'] = pd.to_numeric(district['crimes_95'].replace('?', np.nan), errors='coerce')

In [23]:
print(district[['unemployment_rate_95', 'crimes_95']].isna().sum())

unemployment_rate_95    1
crimes_95               1
dtype: int64


In [24]:
# 1. Impute missing values using the regional median
district['unemployment_rate_95'] = district['unemployment_rate_95'].fillna(
    district.groupby('region')['unemployment_rate_95'].transform('median')
)

district['crimes_95'] = district['crimes_95'].fillna(
    district.groupby('region')['crimes_95'].transform('median')
)

# 2. Cast crimes_95 back to int64
district['crimes_95'] = district['crimes_95'].round().astype('int64')

# 3. Verify zero nulls remain across both columns
print("Missing values after imputation:")
print(district[['unemployment_rate_95', 'crimes_95']].isna().sum())

Missing values after imputation:
unemployment_rate_95    0
crimes_95               0
dtype: int64


In [25]:
district.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77 entries, 0 to 76
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   district_id               77 non-null     int64  
 1   district_name             77 non-null     object 
 2   region                    77 non-null     object 
 3   population                77 non-null     int64  
 4   municipalities_lt_500     77 non-null     int64  
 5   municipalities_500_1999   77 non-null     int64  
 6   municipalities_2000_9999  77 non-null     int64  
 7   municipalities_gt_10000   77 non-null     int64  
 8   num_cities                77 non-null     int64  
 9   urban_ratio               77 non-null     float64
 10  avg_salary                77 non-null     int64  
 11  unemployment_rate_95      77 non-null     float64
 12  unemployment_rate_96      77 non-null     float64
 13  entrepreneurs_per_1000    77 non-null     int64  
 14  crimes_95   

In [26]:
district.to_csv("../cleanData/clean_district.csv", sep=";", index=False)